In [1]:
import can
import struct
import time
import logging

In [ ]:
# Identify PCAN
import can
import time

def test_pcan():
    candidates = ['PCAN_USBBUS1', 'PCAN_USBBUS2', 'PCAN_USBBUS3', 'PCAN_USBBUS4', 'PCAN_USBBUS5', 'PCAN_USBBUS6', 'PCAN_USBBUS7', 'PCAN_USBBUS8', 'PCAN_USBBUS9']
    for ch in candidates:
        try:
            bus = can.Bus(interface='pcan', channel=ch, bitrate=250000)
            print(f"成功连接到 {ch}")
            # 尝试发送一条数据（可选）
            msg = can.Message(arbitration_id=0x123, data=[0x11, 0x22], is_extended_id=True)
            bus.send(msg)
            print("消息发送成功（若硬件未连接终端，可能报错，但至少表示总线已打开）")
            bus.shutdown()
        except Exception as e:
            print(f"通道 {ch} 失败: {e}")
    return False

if __name__ == "__main__":
    if test_pcan():
        print("PCAN 可用")
    else:
        print("PCAN 不可用，请检查驱动或硬件")

In [ ]:
# Initiate BIC2200 Class
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("BIC2200")

class BIC2200:
    """MEAN WELL BIC-2200 双向电源 CANbus 控制类"""

    # 命令码（十六进制）
    CMD_OPERATION              = 0x0000
    CMD_VOUT_SET               = 0x0020
    CMD_IOUT_SET               = 0x0030
    CMD_FAULT_STATUS           = 0x0040
    CMD_READ_VIN               = 0x0050
    CMD_READ_VOUT              = 0x0060
    CMD_READ_IOUT              = 0x0061
    CMD_READ_TEMP1             = 0x0062
    CMD_READ_FAN1              = 0x0070
    CMD_READ_FAN2              = 0x0071
    CMD_MFR_ID_B0B5            = 0x0080
    CMD_MFR_ID_B6B11           = 0x0081
    CMD_MFR_MODEL_B0B5         = 0x0082
    CMD_MFR_MODEL_B6B11        = 0x0083
    CMD_MFR_REVISION           = 0x0084
    CMD_MFR_LOCATION           = 0x0085
    CMD_MFR_DATE               = 0x0086
    CMD_MFR_SERIAL_B0B5        = 0x0087
    CMD_MFR_SERIAL_B6B11       = 0x0088
    CMD_SCALING_FACTOR         = 0x00C0
    CMD_SYSTEM_STATUS          = 0x00C1
    CMD_SYSTEM_CONFIG          = 0x00C2
    CMD_DIRECTION_CTRL         = 0x0100
    CMD_REVERSE_VOUT_SET       = 0x0120
    CMD_REVERSE_IOUT_SET       = 0x0130
    CMD_BIDIRECTIONAL_CONFIG   = 0x0140

    # CAN 消息 ID 基址
    MSGID_CONTROLLER_TO_BIC    = 0x000C0300   # 低 8 位为地址
    MSGID_BIC_TO_CONTROLLER    = 0x000C0200
    MSGID_BROADCAST            = 0x000C03FF

    # 默认因子（可通过 SCALING_FACTOR 读取，此处硬编码）
    FACTOR_VOUT = 0.01
    FACTOR_IOUT = 0.01
    FACTOR_VIN  = 0.1
    FACTOR_TEMP = 0.1
    FACTOR_FAN  = 1.0

    def __init__(self, interface='pcan', channel='PCAN_USBBUS1', address=0, bitrate=250000, timeout=1.0):
        """
        初始化 CAN 接口并设置设备地址。
        :param channel: CAN 接口名称（如 'can0', 'PCAN_USBBUS1'）
        :param address: 设备地址 (0~7)
        :param bitrate: CAN 波特率，默认 250k
        :param timeout: 读取响应超时（秒）
        """
        self.address = address & 0x07
        self.timeout = timeout
        self.bus = can.interface.Bus(channel=channel, bustype=interface, bitrate=bitrate)
        # 构造发送和接收的 CAN ID
        self.tx_id = self.MSGID_CONTROLLER_TO_BIC | self.address
        self.rx_id = self.MSGID_BIC_TO_CONTROLLER | self.address

        # 尝试读取因子（可选），若失败则使用默认值
        try:
            self._update_scaling_factors()
        except Exception as e:
            logger.warning(f"读取因子失败，使用默认值: {e}")

    def _update_scaling_factors(self):
        """从设备读取 SCALING_FACTOR 并更新因子"""
        resp = self._read_command(self.CMD_SCALING_FACTOR, 6)
        if resp and len(resp) >= 6:
            # 解析字节，具体格式参见手册
            # Byte0 低4位 VOUT因子，高4位 IOUT因子
            # Byte1 低4位 VIN因子，高4位 FAN因子
            # Byte2 低4位 TEMP因子
            # 因子编码: 0=不支持, 4=0.001, 5=0.01, 6=0.1, 7=1.0, 8=10, 9=100
            def decode_factor(val):
                mapping = {0: None, 4:0.001, 5:0.01, 6:0.1, 7:1.0, 8:10, 9:100}
                return mapping.get(val, None)
            vout_f = decode_factor(resp[0] & 0x0F)
            iout_f = decode_factor((resp[0] >> 4) & 0x0F)
            vin_f  = decode_factor(resp[1] & 0x0F)
            fan_f  = decode_factor((resp[1] >> 4) & 0x0F)
            temp_f = decode_factor(resp[2] & 0x0F)
            if vout_f is not None: self.FACTOR_VOUT = vout_f
            if iout_f is not None: self.FACTOR_IOUT = iout_f
            if vin_f  is not None: self.FACTOR_VIN  = vin_f
            if fan_f  is not None: self.FACTOR_FAN  = fan_f
            if temp_f is not None: self.FACTOR_TEMP = temp_f
            logger.debug(f"因子更新: VOUT={self.FACTOR_VOUT}, IOUT={self.FACTOR_IOUT}, "
                         f"VIN={self.FACTOR_VIN}, FAN={self.FACTOR_FAN}, TEMP={self.FACTOR_TEMP}")

    def _send_command(self, cmd_code, data=None, wait_response=True, response_len=None):
        """
        发送 CAN 命令（写或读）。
        :param cmd_code: 16位命令码
        :param data: 待发送数据（列表或字节），若为 None 则为读命令
        :param wait_response: 是否等待响应（读命令为 True）
        :param response_len: 期望响应数据长度（不含命令码），若为 None 则自动根据命令判断
        :return: 响应数据（bytes）或 None
        """
        # 构建数据域：命令码低字节在前
        cmd_bytes = struct.pack('<H', cmd_code)  # 小端
        if data is None:
            # 读命令：DLC = 2
            msg_data = cmd_bytes
        else:
            # 写命令：命令码 + 数据
            if isinstance(data, (int, float)):
                # 单个值需转换为字节（通常为2字节）
                data = struct.pack('<H', int(data))
            elif isinstance(data, bytes):
                pass
            else:
                data = bytes(data)
            msg_data = cmd_bytes + data

        msg = can.Message(arbitration_id=self.tx_id, data=msg_data, is_extended_id=True)
        try:
            self.bus.send(msg, timeout=self.timeout)
            logger.debug(f"发送命令 0x{cmd_code:04X} 到地址 {self.address}, 数据: {msg_data.hex()}")
        except can.CanError as e:
            logger.error(f"发送失败: {e}")
            return None

        if not wait_response:
            return None

        # 等待响应（读命令）
        start_time = time.time()
        while time.time() - start_time < self.timeout:
            rx_msg = self.bus.recv(timeout=0.1)
            if rx_msg is None:
                continue
            # 检查是否来自我们的设备且是响应 ID
            if rx_msg.arbitration_id == self.rx_id and rx_msg.is_extended_id:
                # 检查命令码是否匹配
                if len(rx_msg.data) < 2:
                    continue
                resp_cmd = struct.unpack('<H', rx_msg.data[:2])[0]
                if resp_cmd == cmd_code:
                    # 返回数据部分（跳过命令码）
                    resp_data = rx_msg.data[2:]
                    if response_len is not None and len(resp_data) != response_len:
                        logger.warning(f"响应长度不匹配: 期望 {response_len}, 实际 {len(resp_data)}")
                    logger.debug(f"收到响应: {resp_data.hex()}")
                    return resp_data
        logger.error(f"等待响应超时 (命令 0x{cmd_code:04X})")
        return None

    def _read_command(self, cmd_code, expected_len=None):
        """执行读命令并返回数据"""
        return self._send_command(cmd_code, data=None, wait_response=True, response_len=expected_len)

    def _write_command(self, cmd_code, value):
        """执行写命令（带单值）"""
        # value 可以是 int 或 float，转换为整数（乘以因子）
        if isinstance(value, float):
            # 根据命令确定因子
            if cmd_code in (self.CMD_VOUT_SET, self.CMD_REVERSE_VOUT_SET):
                factor = self.FACTOR_VOUT
            elif cmd_code in (self.CMD_IOUT_SET, self.CMD_REVERSE_IOUT_SET):
                factor = self.FACTOR_IOUT
            else:
                factor = 1.0
            int_val = int(round(value / factor))
        else:
            int_val = int(value)
        # 限制范围（不在此处检查）
        data = struct.pack('<H', int_val)
        self._send_command(cmd_code, data=data, wait_response=False)

    # ================= 公共 API =================

    def set_operation(self, on: bool):
        """开启/关闭输出"""
        val = 0x01 if on else 0x00
        self._write_command(self.CMD_OPERATION, val)

    def read_operation(self):
        """读取输出设置"""
        data = self._read_command(self.CMD_OPERATION, 1)
        if data:
            raw = struct.unpack('<B', data)[0]
            return raw

    def set_charge_voltage(self, voltage: float):
        """设置充电电压 (VOUT_SET)"""
        self._write_command(self.CMD_VOUT_SET, voltage)

    def read_charge_voltage(self):
        """读取充电电压 (VOUT_SET)"""
        data = self._read_command(self.CMD_VOUT_SET, 2)
        if data:
            raw = struct.unpack('<H', data)[0]
            return raw * self.FACTOR_VOUT
        return None

    def set_charge_current(self, current: float):
        """设置充电电流 (IOUT_SET)"""
        self._write_command(self.CMD_IOUT_SET, current)

    def set_discharge_voltage(self, voltage: float):
        """设置放电电压 (REVERSE_VOUT_SET)"""
        self._write_command(self.CMD_REVERSE_VOUT_SET, voltage)

    def set_discharge_current(self, current: float):
        """设置放电电流 (REVERSE_IOUT_SET)"""
        # 注意：放电电流为负值，但 CAN 值用正数表示，实际负号由方向决定
        # 手册中 REVERSE_IOUT_SET 的值范围是正数（如 -20A 对应 20A）
        if current < 0:
            current = abs(current)
        self._write_command(self.CMD_REVERSE_IOUT_SET, current)

    def set_direction(self, direction: str):
        """设置方向 (仅电池模式有效) 'charge' 或 'discharge'"""
        if direction.lower() == 'charge':
            val = 0x00
        elif direction.lower() == 'discharge':
            val = 0x01
        else:
            raise ValueError("direction must be 'charge' or 'discharge'")
        self._write_command(self.CMD_DIRECTION_CTRL, val)

    def set_mode(self, mode: str):
        """设置双向模式 'auto' (自动检测) 或 'battery' (电池模式)"""
        if mode.lower() == 'auto':
            val = 0x00
        elif mode.lower() == 'battery':
            val = 0x01
        else:
            raise ValueError("mode must be 'auto' or 'battery'")
        self._write_command(self.CMD_BIDIRECTIONAL_CONFIG, val)
        logger.info("模式变更需要重新上电才能生效。")

    def set_system_config(self, can_ctrl=True, operation_init=0x01, eep_config=0x00, eep_off=False):
        """
        配置系统参数 (SYSTEM_CONFIG)
        :param can_ctrl: True 表示电压/电流由 CAN 控制，False 由 SVR 控制
        :param operation_init: 上电时输出状态 0=OFF, 1=ON, 2=保持上次
        :param eep_config: EEPROM 写入策略 0=立即, 1=延迟1分钟, 2=延迟10分钟
        :param eep_off: True 禁用 EEPROM 存储
        """
        low_byte = 0
        if can_ctrl:
            low_byte |= 0x01
        low_byte |= (operation_init & 0x03) << 1
        high_byte = 0
        high_byte |= (eep_config & 0x03) << 0
        if eep_off:
            high_byte |= (1 << 2)
        config_val = (high_byte << 8) | low_byte
        self._write_command(self.CMD_SYSTEM_CONFIG, config_val)

    # ---------- 读取函数 ----------
    def read_voltage_in(self):
        """读取输入交流电压 (VAC)"""
        data = self._read_command(self.CMD_READ_VIN, 2)
        if data:
            raw = struct.unpack('<H', data)[0]
            return raw * self.FACTOR_VIN
        return None

    def read_voltage_out(self):
        """读取输出直流电压 (VDC)"""
        data = self._read_command(self.CMD_READ_VOUT, 2)
        if data:
            raw = struct.unpack('<H', data)[0]
            return raw * self.FACTOR_VOUT
        return None

    def read_current_out(self):
        """读取输出直流电流（带符号，正=充电，负=放电）"""
        data = self._read_command(self.CMD_READ_IOUT, 2)
        if data:
            raw = struct.unpack('<H', data)[0]
            # 注意：手册未说明符号，但根据逻辑，充电为正，放电为负
            # 但 CAN 读回的值是绝对值还是带符号？文档未明确。
            # 通常 BIC 会返回实际电流方向，可能需要检查 FAULT_STATUS 或系统状态。
            # 这里我们返回原始计算值，用户可根据方向判断。
            return raw * self.FACTOR_IOUT
        return None

    def read_temperature(self):
        """读取内部温度 (℃)"""
        data = self._read_command(self.CMD_READ_TEMP1, 2)
        if data:
            raw = struct.unpack('<H', data)[0]
            return raw * self.FACTOR_TEMP
        return None

    def read_fan_speed_1(self):
        """读取风扇1转速 (RPM)"""
        data = self._read_command(self.CMD_READ_FAN1, 2)
        if data:
            raw = struct.unpack('<H', data)[0]
            return raw * self.FACTOR_FAN
        return None

    def read_fan_speed_2(self):
        """读取风扇2转速 (RPM)"""
        data = self._read_command(self.CMD_READ_FAN2, 2)
        if data:
            raw = struct.unpack('<H', data)[0]
            return raw * self.FACTOR_FAN
        return None

    def read_fault_status(self):
        """读取故障状态 (FAULT_STATUS) 返回两个字节的原始值"""
        data = self._read_command(self.CMD_FAULT_STATUS, 2)
        if data:
            return struct.unpack('<H', data)[0]
        return None

    def read_system_status(self):
        """读取系统状态 (SYSTEM_STATUS) 返回两个字节的原始值"""
        data = self._read_command(self.CMD_SYSTEM_STATUS, 2)
        if data:
            return struct.unpack('<H', data)[0]
        return None

    def read_manufacturer_info(self):
        """读取制造商信息（示例）"""
        data = self._read_command(self.CMD_MFR_ID_B0B5, 6)
        if data:
            return data.decode('ascii', errors='ignore')
        return None

    # ---------- 关闭 CAN 总线 ----------
    def close(self):
        self.bus.shutdown()
        logger.info("CAN 总线已关闭")

    def __del__(self):
        try:
            self.close()
        except:
            pass

In [2]:
# Initiate BIC2200 Class （Simplified Version - Only Read/Write Commands）
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

class BIC2200:
    """MEAN WELL BIC-2200 双向电源 CANbus 控制类"""
    MSGID_CONTROLLER_TO_BIC    = 0x000C0300   # 低 8 位为地址
    MSGID_BIC_TO_CONTROLLER    = 0x000C0200
    MSGID_BROADCAST            = 0x000C03FF

    def __init__(self, interface='pcan', channel='PCAN_USBBUS1', address=0, bitrate=250000, timeout=1.0):
        """
        初始化 CAN 接口并设置设备地址。
        :param channel: CAN 接口名称（如 'can0', 'PCAN_USBBUS1'）
        :param address: 设备地址 (0~7)
        :param bitrate: CAN 波特率，默认 250k
        :param timeout: 读取响应超时（秒）
        """
        self.address = address & 0x07
        self.timeout = timeout
        self.bus = can.interface.Bus(channel=channel, bustype=interface, bitrate=bitrate)
        # 构造发送和接收的 CAN ID
        self.tx_id = self.MSGID_CONTROLLER_TO_BIC | self.address
        self.rx_id = self.MSGID_BIC_TO_CONTROLLER | self.address

    def _send_command(self, cmd_code, data=None, wait_response=True, response_len=None):
        """
        发送 CAN 命令（写或读）。
        :param cmd_code: 16位命令码
        :param data: 待发送数据（列表或字节），若为 None 则为读命令
        :param wait_response: 是否等待响应（读命令为 True）
        :param response_len: 期望响应数据长度（不含命令码），若为 None 则自动根据命令判断
        :return: 响应数据（bytes）或 None
        """
        # 构建数据域：命令码低字节在前
        cmd_bytes = struct.pack('<H', cmd_code)  # 小端
        if data is None:
            # 读命令：DLC = 2
            msg_data = cmd_bytes
        else:
            # 写命令：命令码 + 数据
            if isinstance(data, (int, float)):
                # 单个值需转换为字节（通常为2字节）
                data = struct.pack('<H', int(data))
            elif isinstance(data, bytes):
                pass
            else:
                data = bytes(data)
            msg_data = cmd_bytes + data

        msg = can.Message(arbitration_id=self.tx_id, data=msg_data, is_extended_id=True)
        try:
            self.bus.send(msg, timeout=self.timeout)
            logger.debug(f"发送命令 0x{cmd_code:04X} 到地址 {self.address}, 数据: {msg_data.hex()}")
        except can.CanError as e:
            logger.error(f"发送失败: {e}")
            return None

        if not wait_response:
            return None

        # 等待响应（读命令）
        start_time = time.time()
        while time.time() - start_time < self.timeout:
            rx_msg = self.bus.recv(timeout=0.1)
            if rx_msg is None:
                continue
            # 检查是否来自我们的设备且是响应 ID
            if rx_msg.arbitration_id == self.rx_id and rx_msg.is_extended_id:
                # 检查命令码是否匹配
                if len(rx_msg.data) < 2:
                    continue
                resp_cmd = struct.unpack('<H', rx_msg.data[:2])[0]
                if resp_cmd == cmd_code:
                    # 返回数据部分（跳过命令码）
                    resp_data = rx_msg.data[2:]
                    if response_len is not None and len(resp_data) != response_len:
                        logger.warning(f"响应长度不匹配: 期望 {response_len}, 实际 {len(resp_data)}")
                    logger.debug(f"收到响应: {resp_data.hex()}")
                    return resp_data
        logger.error(f"等待响应超时 (命令 0x{cmd_code:04X})")
        return None

    def _read_command(self, cmd_code, data_bytes, timeout = 3):
        """执行读命令并返回数据"""
        cmd_code_processed = struct.pack('<H', cmd_code)
        msg_data = cmd_code_processed
        msg = can.Message(arbitration_id=self.tx_id, data=msg_data, is_extended_id=True)
        self.bus.send(msg, timeout=self.timeout)
        # 等待响应（读命令）
        start_time = time.time()
        while time.time() - start_time < timeout:
            rx_msg = self.bus.recv(timeout=0.1)
            if rx_msg is None:
                continue
            # 检查是否来自我们的设备且是响应 ID
            if rx_msg.arbitration_id == self.rx_id and rx_msg.is_extended_id:
				# 检查命令码是否匹配
                if len(rx_msg.data) < 2:
                    continue
                resp_cmd = struct.unpack('<H', rx_msg.data[:2])[0]
                if resp_cmd == cmd_code:
                    # 返回数据部分（跳过命令码）
                    resp_data = rx_msg.data[2:]
                    if data_bytes is not None and len(resp_data) != data_bytes:
                        raise ValueError(f"响应长度不匹配: 期望 {data_bytes}, 实际 {len(resp_data)}")
                    return int.from_bytes(resp_data, byteorder='little')

    def _write_command(self, cmd_code, value, data_bytes):
        """执行写命令（带单值）"""
        # 限制范围（不在此处检查）
        cmd_code_processed = struct.pack('<H', cmd_code)
        if data_bytes == 1:
            value_processed = struct.pack('<B', int(value))
        else:
            value_processed = struct.pack('<H', int(value))
        msg_data = cmd_code_processed + value_processed
        msg = can.Message(arbitration_id=self.tx_id, data=msg_data, is_extended_id=True)
        self.bus.send(msg, timeout=self.timeout)
        #self._send_command(cmd_code, data=value_processed, wait_response=False)

    # ---------- 关闭 CAN 总线 ----------
    def close(self):
        self.bus.shutdown()

    def __del__(self):
        try:
            self.close()
        except:
            pass



In [ ]:
psu = BIC2200(address=0)
psu._write_command(cmd_code=0x0020, value=1200, data_bytes=2)
psu._read_command(cmd_code=0x0020, data_bytes=2)

In [ ]:
# Send command 0x00C2 with data 0x0003 to enable CAN CTRL (Preset ON)
psu._send_command(0x00C2, data=0x0003, wait_response=False)

In [ ]:
# Send command 0x00C2 with data 0x0003 to enable CAN CTRL (Preset OFF)
psu._send_command(0x00C2, data=0x0001, wait_response=False)

In [3]:
channel = 'PCAN_USBBUS2'
psu = BIC2200(address=0, channel = channel)

C:\Users\LiewChuanNyen\AppData\Local\Temp\ipykernel_22096\1705699438.py:21: DeprecationWarning: The 'bustype' argument is deprecated since python-can v4.2.0, and scheduled for removal in python-can v5.0.0. Use 'interface' instead.
  self.bus = can.interface.Bus(channel=channel, bustype=interface, bitrate=bitrate)


In [ ]:
psu.close()

In [5]:
psu._write_command(cmd_code=0x0000, value=1, data_bytes=1)

In [ ]:
psu._write_command(cmd_code=0x00C2, value=1, data_bytes=2)

In [9]:
psu._read_command(cmd_code=0x0020, data_bytes=2)

4800

In [ ]:
psu._write_command(cmd_code=0x0020, value=1500, data_bytes=2)